# Autoresearch Experiment Analysis

Analysis of autonomous classifier tuning results from `results.tsv`.
Task: Real vs. Fake Job Posting Detection — binary text classification.
Primary metric: `val_f1_macro` (higher is better).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_f1_macro, val_pr_auc, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_f1_macro"] = pd.to_numeric(df["val_f1_macro"], errors="coerce")
df["val_pr_auc"] = pd.to_numeric(df["val_pr_auc"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
    kept = df[df["status"] == "KEEP"]
    best = kept["val_f1_macro"].max()
    print(f"Best val_f1_macro so far: {best:.6f}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    f1 = row["val_f1_macro"]
    pr_auc = row["val_pr_auc"]
    desc = row["description"]
    print(f"  #{i:3d}  f1_macro={f1:.6f}  pr_auc={pr_auc:.6f}  {desc}")

## Val F1-Macro Over Time

Track how the best (kept) val_f1_macro evolves as experiments progress. The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_f1 = valid.loc[0, "val_f1_macro"]
best_f1 = valid.loc[valid["status"] == "KEEP", "val_f1_macro"].max()

# Only plot points at or above baseline (the interesting region)
above = valid[valid["val_f1_macro"] >= baseline_f1 - 0.005]

# Plot discarded as faint background dots
disc = above[above["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_f1_macro"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = above[above["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_f1_macro"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line (higher is better)
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_f1 = valid.loc[kept_mask, "val_f1_macro"]
running_max = kept_f1.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, f1 in zip(kept_idx, kept_f1):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, f1),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation F1-Macro (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
margin = max((best_f1 - baseline_f1) * 0.15, 0.01)
ax.set_ylim(baseline_f1 - margin, best_f1 + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_f1 = df.iloc[0]["val_f1_macro"]
best_f1 = kept["val_f1_macro"].max()
best_row = kept.loc[kept["val_f1_macro"].idxmax()]

print(f"Baseline val_f1_macro:  {baseline_f1:.6f}")
print(f"Best val_f1_macro:      {best_f1:.6f}")
print(f"Total improvement:      {best_f1 - baseline_f1:+.6f} ({(best_f1 - baseline_f1) / max(baseline_f1, 1e-9) * 100:.2f}%)")
print(f"Best experiment:        {best_row['description']}")
print()

# PR-AUC progression
baseline_pr = df.iloc[0]["val_pr_auc"]
best_pr = kept["val_pr_auc"].max()
print(f"Baseline val_pr_auc:    {baseline_pr:.6f}")
print(f"Best val_pr_auc:        {best_pr:.6f}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: f1_macro={row['val_f1_macro']:.6f}  pr_auc={row['val_pr_auc']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's f1_macro
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_f1"] = kept["val_f1_macro"].shift(1)
kept["delta"] = kept["val_f1_macro"] - kept["prev_f1"]  # positive = improvement

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'F1-Macro':>10}  {'PR-AUC':>8}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_f1_macro']:.6f}  {row['val_pr_auc']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>8}  TOTAL improvement over baseline")